# Klassifikationsbäume: Kundenkündigung vorhersagen

**Interaktive Colab-Übung (ohne Programmierkenntnisse)** · synthetischer Datensatz mit **60 Kunden**.

**Lernziele**
1. Eingabevariablen (**Vertragsdauer**, **Beschwerden**) und Zielvariable (**Kündigung: Ja/Nein**) unterscheiden.
2. Daten im Scatterplot untersuchen und einen **Klassifikationsbaum** trainieren.
3. **Gini** und **Entropy** als Maße der Unreinheit vergleichen.
4. **Kündigungswahrscheinlichkeiten aus Blatt-Häufigkeiten** ableiten und neue Kunden klassifizieren.
5. Training/Test sowie zu komplexe Bäume kritisch beurteilen.

▶️ In Colab: **Laufzeit → Alle ausführen**; danach die Regler und Dropdowns ändern.

*Datengrundlage: der für diese Vorlesung erzeugte fiktive Excel-Datensatz. Es sind keine echten Kundendaten.*

## 1 · Datensatz laden
Du kannst die frühere Datei `Kundenkuendigung_mit_Scatterplot.xlsx` hochladen. **Ohne Upload** benutzt das Notebook dieselben 60 Beobachtungen direkt aus dieser Datei (integrierte Kopie), damit die Übung sofort funktioniert.

In [ ]:
%pip -q install ipywidgets
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from IPython.display import display, Markdown
import ipywidgets as widgets

# Daten sind unverändert aus der Excel-Datei der Vorlesung eingebettet.
columns = ["Kunden-ID", "Vertragsdauer (Monate)", "Beschwerden (6 Monate)", "Monatsbetrag (€)", "Kündigung (Ja/Nein)"]
embedded_rows = [["K038", 2, 0, 87.31, "Nein"], ["K046", 2, 0, 108.83, "Nein"], ["K006", 2, 4, 45.3, "Ja"], ["K045", 2, 5, 56.2, "Ja"], ["K034", 4, 0, 41.3, "Nein"], ["K026", 4, 2, 62.9, "Ja"], ["K059", 4, 2, 34.16, "Ja"], ["K017", 6, 0, 85.53, "Nein"], ["K048", 6, 0, 91.68, "Nein"], ["K058", 6, 0, 47.92, "Nein"], ["K027", 6, 1, 107.44, "Nein"], ["K056", 6, 2, 29.83, "Ja"], ["K049", 6, 4, 94.63, "Ja"], ["K022", 6, 5, 83.24, "Ja"], ["K001", 8, 0, 92.51, "Nein"], ["K012", 8, 2, 101.73, "Nein"], ["K042", 8, 2, 101.23, "Ja"], ["K014", 8, 3, 34.86, "Ja"], ["K019", 8, 3, 52.18, "Ja"], ["K003", 8, 4, 35.56, "Ja"], ["K010", 10, 1, 111.28, "Nein"], ["K030", 10, 1, 92.81, "Nein"], ["K050", 10, 3, 110.38, "Ja"], ["K033", 10, 4, 70.94, "Ja"], ["K051", 12, 2, 73.91, "Ja"], ["K009", 12, 3, 57.6, "Ja"], ["K035", 12, 3, 79.89, "Ja"], ["K044", 12, 4, 110.95, "Nein"], ["K005", 15, 1, 71.97, "Nein"], ["K043", 15, 1, 60.53, "Nein"], ["K021", 15, 2, 89.06, "Nein"], ["K053", 15, 2, 62.71, "Ja"], ["K002", 18, 1, 40.14, "Nein"], ["K023", 18, 1, 68.22, "Nein"], ["K060", 18, 2, 86.2, "Nein"], ["K057", 18, 5, 47.16, "Nein"], ["K008", 22, 0, 94.02, "Nein"], ["K013", 22, 0, 91.49, "Nein"], ["K039", 22, 2, 37.71, "Nein"], ["K024", 22, 4, 47.11, "Nein"], ["K041", 22, 4, 94.29, "Nein"], ["K031", 22, 5, 65.27, "Nein"], ["K018", 26, 0, 102.41, "Nein"], ["K047", 26, 1, 33.04, "Nein"], ["K015", 26, 5, 105.03, "Nein"], ["K011", 30, 0, 36.07, "Nein"], ["K025", 30, 0, 47.93, "Nein"], ["K028", 30, 1, 85.02, "Nein"], ["K020", 36, 1, 60.21, "Nein"], ["K054", 36, 3, 106.27, "Nein"], ["K016", 36, 5, 44.73, "Nein"], ["K032", 42, 2, 47.08, "Nein"], ["K029", 42, 3, 40.43, "Ja"], ["K036", 42, 3, 79.84, "Nein"], ["K004", 48, 0, 30.59, "Nein"], ["K007", 48, 1, 67.08, "Nein"], ["K040", 48, 1, 67.47, "Nein"], ["K052", 48, 1, 108.82, "Nein"], ["K055", 54, 0, 49.57, "Nein"], ["K037", 54, 4, 49.87, "Nein"]]
df = pd.DataFrame(embedded_rows, columns=columns)

# Optional: eigene Excel-Datei hochladen (in Colab diese drei Zeilen aktivieren):
# from google.colab import files
# uploaded = files.upload()
# df = pd.read_excel(io.BytesIO(next(iter(uploaded.values()))), sheet_name="Kundendaten")

XCOLS = ["Vertragsdauer (Monate)", "Beschwerden (6 Monate)"]
YCOL = "Kündigung (Ja/Nein)"
assert len(df)==60 and set(df[YCOL].unique())=={"Ja","Nein"}
display(df.head(10))
print(f"{len(df)} Beobachtungen · {sum(df[YCOL]=='Ja')} Kündigungen · {sum(df[YCOL]=='Nein')} Nicht-Kündigungen")

## 2 · Scatterplot: Wo befinden sich die beiden Klassen?

**Fragen:** Welche Merkmale lassen Kündigungen erwarten? Wo würdest du eine erste Trennlinie ziehen?

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
for klasse, color, marker in [("Ja","#17658a","D"),("Nein","#e87532","s")]:
    part=df[df[YCOL]==klasse]
    ax.scatter(part[XCOLS[0]],part[XCOLS[1]],label=f"Kündigung: {klasse}",color=color, marker=marker, s=55, alpha=.85)
ax.set(xlabel="Vertragsdauer (Monate)", ylabel="Beschwerden in 6 Monaten", title="Kundenkündigung: Zwei Eingabevariablen")
ax.set_ylim(-.5,6);ax.grid(alpha=.2);ax.legend();plt.show()

## 3 · Einen Klassifikationsbaum trainieren

Verändere **Kriterium**, **maximale Baumtiefe** und **Mindestanzahl pro Blatt**. Die Diagramme zeigen den *auf Trainingsdaten gelernten Baum* und seine Vorhersagebereiche.

Die Daten werden reproduzierbar aufgeteilt: **80 % Training / 20 % Test**. Der Testdatensatz wird **nicht** zum Fitten verwendet.

In [ ]:
X = df[XCOLS]
y = df[YCOL]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.20, random_state=42, stratify=y)

criterion = widgets.Dropdown(options=[("Gini", "gini"),("Entropy", "entropy")],value="gini",description="Kriterium:",style={"description_width":"initial"})
depth = widgets.IntSlider(value=2,min=1,max=8,description="Max. Tiefe:",continuous_update=False,style={"description_width":"initial"})
leaf = widgets.IntSlider(value=3,min=1,max=12,description="Min. pro Blatt:",continuous_update=False,style={"description_width":"initial"})

def fit_and_show(criterion, depth, leaf):
    model=DecisionTreeClassifier(criterion=criterion,max_depth=depth,min_samples_leaf=leaf,random_state=42)
    model.fit(X_train,y_train)
    fig,axes=plt.subplots(1,2,figsize=(16,5.7))
    plot_tree(model,feature_names=["Vertragsdauer","Beschwerden"],class_names=list(model.classes_),
              filled=True,rounded=True,impurity=True,proportion=False,precision=2,fontsize=9,ax=axes[0])
    axes[0].set_title("Entscheidungsbaum (Trainingsdaten)")
    xx,yy=np.meshgrid(np.linspace(0,45,220),np.linspace(-.3,5.5,175))
    grid=pd.DataFrame({XCOLS[0]:xx.ravel(),XCOLS[1]:yy.ravel()})
    z=(model.predict(grid)=="Ja").astype(int).reshape(xx.shape)
    axes[1].contourf(xx,yy,z,levels=[-.5,.5,1.5],colors=["#ffd8bd","#cfeaf7"],alpha=.6)
    for k,c,m in [("Ja","#17658a","D"),("Nein","#e87532","s")]:
        part=df[df[YCOL]==k]
        axes[1].scatter(part[XCOLS[0]],part[XCOLS[1]],label=f"Tatsächlich: {k}",c=c,marker=m,s=39,edgecolor="white",linewidth=.4)
    axes[1].set(xlabel="Vertragsdauer (Monate)",ylabel="Beschwerden (6 Monate)",xlim=(0,45),ylim=(-.3,5.5),title="Vorhergesagte Entscheidungsbereiche")
    axes[1].legend(loc="upper right",fontsize=9)
    plt.tight_layout();plt.show()
    print("Genauigkeit auf Trainingsdaten:",f"{accuracy_score(y_train,model.predict(X_train)):.1%}")
    print("Genauigkeit auf Testdaten:     ",f"{accuracy_score(y_test,model.predict(X_test)):.1%}")
    print("Blätter:",model.get_n_leaves()," | Baumtiefe:",model.get_depth())
    return model

out = widgets.interactive_output(fit_and_show,{"criterion":criterion,"depth":depth,"leaf":leaf})
display(widgets.HBox([criterion,depth,leaf]),out)

## 4 · Gini und Entropy selbst berechnen

Für jeden Knoten gilt mit \(p_{Ja}\) und \(p_{Nein}\):

\[\mathrm{Gini}=1-p_{Ja}^2-p_{Nein}^2\]

\[\mathrm{Entropy}=-p_{Ja}\log_2(p_{Ja})-p_{Nein}\log_2(p_{Nein})\]

Bei \(p=0\) wird der Term \(p\log_2(p)\) als **0** definiert. Niedrigere Werte bedeuten reinere Knoten; die Skalen unterscheiden sich.

In [ ]:
def gini_entropy(n_yes, n_no):
    n=n_yes+n_no
    probs=[n_yes/n,n_no/n]
    gini=1-sum(p*p for p in probs)
    entropy=-sum(p*np.log2(p) for p in probs if p>0)
    return gini,entropy

# Vordefinierte Regeln aus der Einstiegsübung, auf ALLE 60 Daten angewandt.
groups={
    "Vertragsdauer ≤ 18 & Beschwerden ≥ 2":df[(df[XCOLS[0]]<=18)&(df[XCOLS[1]]>=2)],
    "Vertragsdauer ≤ 18 & Beschwerden < 2":df[(df[XCOLS[0]]<=18)&(df[XCOLS[1]]<2)],
    "Vertragsdauer > 18":df[df[XCOLS[0]]>18]
}
records=[]
for name,group in groups.items():
    yes=sum(group[YCOL]=="Ja");no=len(group)-yes
    g,h=gini_entropy(yes,no)
    records.append([name,len(group),yes,no,round(yes/len(group),3),round(g,3),round(h,3)])
display(pd.DataFrame(records,columns=["Bereich","Kunden","Ja","Nein","P(Kündigung)","Gini","Entropy"]))
print("Beispiel: 17 Kündigungen bei 22 Kunden = 17/22 = 77,3 % Wahrscheinlichkeit")
print("ACHTUNG: Diese handgewählten Regeln müssen NICHT dem tatsächlich gelernten Baum entsprechen.")

### Wie wählt der Algorithmus einen Split?
Der Algorithmus berechnet die gewichtete Unreinheit der beiden Kindknoten und vergleicht sie mit dem Elternknoten. Eine größere **Reduktion der Unreinheit** ist besser.

In [ ]:
# Beispiel: Kandidat "Vertragsdauer ≤ 18" auf TRAININGSDATEN (damit keine Leckage entsteht)
mask=X_train[XCOLS[0]]<=18
parent_yes=sum(y_train=="Ja");parent_no=len(y_train)-parent_yes
left_yes=sum(y_train[mask]=="Ja");left_no=sum(mask)-left_yes
right_yes=sum(y_train[~mask]=="Ja");right_no=sum(~mask)-right_yes
for metric,ix in [("Gini",0),("Entropy",1)]:
    parent=gini_entropy(parent_yes,parent_no)[ix]
    left=gini_entropy(left_yes,left_no)[ix]
    right=gini_entropy(right_yes,right_no)[ix]
    weighted=(sum(mask)*left+sum(~mask)*right)/len(y_train)
    print(f"{metric}: Eltern={parent:.3f}, linker Knoten={left:.3f}, rechter Knoten={right:.3f}, gewichtete Unreinheit={weighted:.3f}, Verbesserung={parent-weighted:.3f}")

## 5 · Einen neuen Kunden vorhersagen

Stelle **Vertragsdauer** und **Anzahl der Beschwerden** ein. Das Modell wird mit den oben gewählten Einstellungen **nur auf Trainingsdaten** gefittet. Danach wird die geschätzte Wahrscheinlichkeit aus dem erreichten Endknoten ausgegeben.

In [ ]:
months=widgets.IntSlider(value=8,min=0,max=45,description="Vertragsdauer:",style={"description_width":"initial"})
complaints=widgets.IntSlider(value=3,min=0,max=5,description="Beschwerden:",style={"description_width":"initial"})

def predict_one(months,complaints,criterion,depth,leaf):
    model=DecisionTreeClassifier(criterion=criterion,max_depth=depth,min_samples_leaf=leaf,random_state=42).fit(X_train,y_train)
    customer=pd.DataFrame([[months,complaints]],columns=XCOLS)
    probs=dict(zip(model.classes_,model.predict_proba(customer)[0]))
    node_id=model.apply(customer)[0]
    counts=model.tree_.value[node_id][0]
    count_map=dict(zip(model.classes_,counts))
    print(f"Neuer Kunde: {months} Monate, {complaints} Beschwerden")
    print(f"Vorhersage: Kündigung = {model.predict(customer)[0]}")
    print(f"P(Kündigung: Ja)   = {probs['Ja']:.1%}")
    print(f"P(Kündigung: Nein) = {probs['Nein']:.1%}")
    print("Endknoten: Klassenanteile der Trainingsfälle",{k:round(float(v),3) for k,v in count_map.items()})

interactive_prediction=widgets.interactive_output(predict_one,{"months":months,"complaints":complaints,"criterion":criterion,"depth":depth,"leaf":leaf})
display(widgets.VBox([months,complaints]),interactive_prediction)

## 6 · Reflexion und Prüfung des Modells

- **Warum unterscheiden sich Gini und Entropy zahlenmäßig?** Sie verwenden unterschiedliche Formeln und Skalen, dienen aber demselben Ziel: möglichst reine Gruppen zu erzeugen.
- **Was passiert bei größerer Baumtiefe?** Meist wird die Trainingsanpassung besser; die Testgenauigkeit muss sich nicht verbessern (*Overfitting*).
- **Sind Blattwahrscheinlichkeiten echte Kündigungsrisiken?** Nein, zunächst handelt es sich um empirische Klassenanteile. Bei wenigen Fällen können sie stark schwanken; eine Kalibrierung wäre für verlässliche Wahrscheinlichkeiten wichtig.
- **Warum nur zwei Merkmale?** So ist der Entscheidungsraum zweidimensional sichtbar. In der Praxis können weitere Variablen wichtig sein.

**Hinweis für Lehrende:** Der visuelle Beispielbaum aus der Einführung enthält manuell gewählte Schwellen. Der automatisch gelernte Baum kann andere Splits erzeugen. Die Tabelle mit 17/22, 0/14 und 1/24 nutzt sämtliche 60 Beobachtungen; die interaktive Modellbewertung dagegen nur 48 Trainingsbeobachtungen und 12 Testbeobachtungen.